In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
RESULTS_DIR = PROJECT_DIR / "results" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)


Project directory: c:\Users\ASUS\Documents\GitHub\Communication-Network-Failure-and-Resilience
Raw data directory: c:\Users\ASUS\Documents\GitHub\Communication-Network-Failure-and-Resilience\data\raw


In [7]:
PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "data" / "raw"

print("Current directory:", Path.cwd())
print("Raw data directory:", RAW_DIR)
print("\nFiles found in data/raw:")

if RAW_DIR.exists():
    for file in RAW_DIR.iterdir():
        print(file.name)
else:
    print("The data/raw folder does not exist!")
    

edges_file = RAW_DIR / "email-Eu-core.txt.gz"
departments_file = RAW_DIR / "email-Eu-core-department-labels.txt.gz"

print("Connections file exists:", edges_file.exists())
print("Department labels file exists:", departments_file.exists())
    


Current directory: c:\Users\ASUS\Documents\GitHub\Communication-Network-Failure-and-Resilience\notebooks
Raw data directory: c:\Users\ASUS\Documents\GitHub\Communication-Network-Failure-and-Resilience\data\raw

Files found in data/raw:
email-Eu-core-department-labels.txt.gz
email-Eu-core.txt.gz
Connections file exists: True
Department labels file exists: True


In [12]:
edges = pd.read_csv(
    edges_file,
    sep=r"\s+",
    header=None,
    names=["source", "target"],
    dtype={"source": "int64", "target": "int64"}
)

print("Connections dataset shape:", edges.shape)
display(edges.head())
display(edges.info())
print("Total connections:", len(edges))


Connections dataset shape: (25571, 2)


,source,target
0,0,1
1,2,3
2,2,4
3,5,6
4,5,7


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25571 entries, 0 to 25570
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   source  25571 non-null  int64
 1   target  25571 non-null  int64
dtypes: int64(2)
memory usage: 399.7 KB


None

Total connections: 25571


In [13]:
departments = pd.read_csv(
    departments_file,
    sep=r"\s+",
    header=None,
    names=["node", "department"],
    dtype={"node": "int64", "department": "int64"}
)

print("Department dataset shape:", departments.shape)
display(departments.head())
display(departments.info())


Department dataset shape: (1005, 2)


,node,department
0,0,1
1,1,1
2,2,21
3,3,21
4,4,21


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1005 entries, 0 to 1004
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   node        1005 non-null   int64
 1   department  1005 non-null   int64
dtypes: int64(2)
memory usage: 15.8 KB


None

In [17]:
print("Missing values in connections:")
display(edges.isnull().sum())

print("\nMissing values in department labels:")
display(departments.isnull().sum())

print("\nDuplicate connection rows:", edges.duplicated().sum())
print("Duplicate department-label rows:", departments.duplicated().sum())

print("\nSelf-loop connections:", (edges["source"] == edges["target"]).sum())

print("\nUnique source nodes:", edges["source"].nunique())
print("Unique target nodes:", edges["target"].nunique())
print("Unique department-labelled nodes:", departments["node"].nunique())
print("Number of departments:", departments["department"].nunique())


Missing values in connections:


source    0
target    0
dtype: int64


Missing values in department labels:


node          0
department    0
dtype: int64


Duplicate connection rows: 0
Duplicate department-label rows: 0

Self-loop connections: 642

Unique source nodes: 868
Unique target nodes: 991
Unique department-labelled nodes: 1005
Number of departments: 42


In [18]:
edge_nodes = set(edges["source"]) | set(edges["target"])
labelled_nodes = set(departments["node"])

print("Unique nodes appearing in connections:", len(edge_nodes))
print("Unique nodes with department labels:", len(labelled_nodes))

print("Nodes in connections without department labels:",
      len(edge_nodes - labelled_nodes))

print("Department labels without connections:",
      len(labelled_nodes - edge_nodes))

print("Duplicate node IDs in department file:",
      departments["node"].duplicated().sum())


Unique nodes appearing in connections: 1005
Unique nodes with department labels: 1005
Nodes in connections without department labels: 0
Department labels without connections: 0
Duplicate node IDs in department file: 0


In [20]:
edges_clean = edges.drop_duplicates().copy()

# A node should have only one department label.
# If duplicates exist, inspect them rather than guessing a label.
if departments["node"].duplicated().any():
    raise ValueError(
        "Some nodes have multiple department-label rows. "
        "Inspect these records before continuing."
    )

departments_clean = departments.copy()


# Save the cleaned datasets
edges_clean.to_csv(
    PROCESSED_DIR / "email_edges_clean.csv",
    index=False
)

departments_clean.to_csv(
    PROCESSED_DIR / "email_departments_clean.csv",
    index=False
)

print("Cleaned files saved successfully.")
print("Clean connections:", edges_clean.shape)
print("Clean department labels:", departments_clean.shape)


Cleaned files saved successfully.
Clean connections: (25571, 2)
Clean department labels: (1005, 2)


In [21]:
summary = pd.DataFrame({
    "Metric": [
        "Original connection rows",
        "Clean connection rows",
        "Duplicate connection rows removed",
        "Unique nodes in connections",
        "Department label rows",
        "Unique departments",
        "Nodes missing department labels"
    ],
    "Value": [
        len(edges),
        len(edges_clean),
        len(edges) - len(edges_clean),
        len(edge_nodes),
        len(departments_clean),
        departments_clean["department"].nunique(),
        len(edge_nodes - labelled_nodes)
    ]
})

display(summary)

summary.to_csv(
    RESULTS_DIR / "data_validation_summary.csv",
    index=False
)


,Metric,Value
0,Original connection rows,25571
1,Clean connection rows,25571
2,Duplicate connection rows removed,0
3,Unique nodes in connections,1005
4,Department label rows,1005
5,Unique departments,42
6,Nodes missing department labels,0
